# 13 · How fast the criterion follows the distribution

**What this shows.** Chapter 3's Fig 3.6: the manuscript's Fig. 5C measure, made steadier, for both Hard designs —
the blocks of days of the pre-opto mice and the daily alternation of the opto mice (masking days; laser days, on
laser-off trials, as a check). After a switch the criterion moves from where it was (0) towards the normative
criterion of the new distribution (1). One number per mouse: the average over trials 100–300 after the switch,
inside the first day, where the slow drift across days hardly enters. Then where the blocks end, read through the
Hard-A → Hard-B switch (the drift works against it), and the overnight check.

**How it is measured** (`behaviour/adaptation.py`). The criterion is the 50 % point of the psychometric fitted with
slope and lapses fixed at the mouse's expert-Uniform fit, in running 50-trial windows; a fit that ends at ±1 (the
mouse chose mostly one side) is missing. 0 = the criterion over the last 250 trials before the switch, 1 = the
normative criterion at the mouse's sigma; not clipped. A mouse's switches of one kind are combined weighted by
|normative − before| (the distance moved over the distance to go), then the mice are averaged with a 95 % t
interval. Mice the bias check flags (`bias.csv`, the same list in every report) are left out.

**What produced it.** Two `switch_adaptation` runs:

```bash
sc-reports switch-adaptation --cohort behaviour1-cohort
sc-reports switch-adaptation --cohort opto1-cohort
```

In [ ]:
from nb_setup import *
from sound_categorisation.behaviour.adaptation import pool_convergence
from sound_categorisation.reports.switches import BLOCK_KINDS, DAY_KINDS, pooled_curve, t_interval

banner()
runs = {c: open_run('switch_adaptation', c) for c in dict.fromkeys([SWITCH_COHORT, OPTO_COHORT])}
tables = [load_switches(r) for r in runs.values()]


def read(name):
    frames = [t[name] for t in tables if name in t and len(t[name])]
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def clean(df):
    return df[~df['animal'].isin(biased)] if len(df) else df


conv, animals, switches, bias, overnight = (read(n) for n in ('convergence', 'animals', 'switches', 'bias', 'overnight'))
biased = sorted(set(bias.loc[bias['biased_animal'].astype(bool), 'animal']))
LABELS = ['animal', 'switch_idx', 'from_distribution', 'to_distribution', 'transition', 'design', 'switch_kind']
wide = switches.groupby(LABELS + ['stat'])['value'].first().unstack('stat').reset_index()
print('left out by the bias check:', biased or 'none')
animals.groupby(['design', 'switch_kind']).size()

## 1 · One mouse through the pipeline

`compute_switches` finds the mouse's switches (`hard_design` decides whether they are blocks or days) and measures
each one: the criterion before the switch, the curve after it, `convergence_100_300` and where the block ends.

In [ ]:
from sound_categorisation.behaviour.adaptation import compute_switches, hard_design

exp = load_experiment()
candidates = sorted(set(animals.loc[animals['design'] == 'blocked', 'animal']) - set(biased)) or sorted(set(animals['animal']))
aid = candidates[0]
kw = dict(min_block_trials=300, max_trials=800) if SYNTHETIC else {}
res = compute_switches(exp.animals[aid], **kw)
print(aid, hard_design(exp.animals[aid]), *res, sep='\n')
r = res[min(1, len(res) - 1)]
r.convergence[r.convergence['method'] == 'pinned_running'].head(8).round(3)

## 2 · The figure

Left: the criterion after a switch, each kind of switch averaged over the clean mice (band = 95 % t interval); the
grey band marks trials 100–300. Middle: one number per mouse, the mean over trials 100–300, with the mean over mice
and its 95 % t interval. Right: where the Hard-A → Hard-B block ends (its last 250 trials). The slow upward drift
across days works against this switch, so a mouse that ends above 0 moved with the distribution despite it; one
near 0 tells us nothing either way.

In [ ]:
kinds = [('blocked', k) for k in BLOCK_KINDS] + [('daily', k) for k in DAY_KINDS]
colours = {'first': '0.55', 'novel': PALETTE[0], 'return': PALETTE[1], 'masking day': PALETTE[2], 'laser day': PALETTE[3]}
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), gridspec_kw={'width_ratios': [1.4, 1, 0.6]})

ax = axes[0]
cv = clean(conv[conv['method'] == 'pinned_running'])
for design, kind in kinds:
    per = pooled_curve(cv[(cv['design'] == design) & (cv['switch_kind'] == kind)])
    if not len(per):
        continue
    g = pd.DataFrame([(t, *t_interval(d['convergence'])) for t, d in per.groupby('trial')],
                     columns=['trial', 'mean', 'lo', 'hi', 'n'])
    ax.plot(g['trial'], g['mean'], color=colours[kind], lw=2, label=f'{kind} ({design}, n = {int(g["n"].max())})')
    ax.fill_between(g['trial'], g['lo'], g['hi'], color=colours[kind], alpha=0.15, lw=0)
ax.axvspan(100, 300, color='0.93', zorder=0)
ax.axhline(1, ls='--', color='k', lw=0.9)
ax.axhline(0, color='0.6', lw=0.8)
ax.set(xscale='log', xlim=(20, 3000), ylim=(-1, 2), xlabel='trials after the switch (window centre, log)',
       ylabel='convergence (0 = before, 1 = normative)')
ax.set_xticks([25, 50, 100, 300, 1000, 3000])
ax.set_xticklabels(['25', '50', '100', '300', '1000', '3000'])
ax.minorticks_off()
ax.legend(frameon=False, fontsize=8)

ax = axes[1]
an = clean(animals)
for i, (design, kind) in enumerate(kinds):
    v = an.loc[(an['design'] == design) & (an['switch_kind'] == kind), 'convergence_100_300'].dropna().to_numpy()
    ax.plot(i + (np.linspace(-0.15, 0.15, len(v)) if len(v) > 1 else np.zeros(len(v))), v, 'o', color=colours[kind], alpha=0.75)
    m, lo, hi, n = t_interval(v)
    if np.isfinite(m):
        ax.plot([i - 0.25, i + 0.25], [m, m], color='k', lw=1.6)
    if np.isfinite(lo):
        ax.plot([i, i], [lo, hi], color='k', lw=1.2)
ax.set_xticks(range(len(kinds)))
ax.set_xticklabels([k for _, k in kinds], rotation=30)
ax.axhline(1, ls='--', color='k', lw=0.9)
ax.axhline(0, color='0.6', lw=0.8)
ax.set(ylim=(-1, 2), ylabel='convergence over trials 100-300')

ax = axes[2]
ab = clean(wide[(wide['design'] == 'blocked') & (wide['from_distribution'] == 'Hard-A') & (wide['to_distribution'] == 'Hard-B')])
end = np.array([pool_convergence(g['convergence_final'], g['span']) for _, g in ab.groupby('animal')]) if len(ab) else np.array([])
ax.plot(np.linspace(-0.15, 0.15, len(end)) if len(end) > 1 else np.zeros(len(end)), end, 'o', color=colours['return'], alpha=0.75)
m, lo, hi, n = t_interval(end)
if np.isfinite(m):
    ax.plot([-0.25, 0.25], [m, m], color='k', lw=1.6)
if np.isfinite(lo):
    ax.plot([0, 0], [lo, hi], color='k', lw=1.2)
ax.axhline(1, ls='--', color='k', lw=0.9)
ax.axhline(0, color='0.6', lw=0.8)
ax.set(xlim=(-1, 1), xticks=[0], xticklabels=[f'Hard-A → Hard-B\n(n = {n})'], ylabel='convergence at the end of the block')
plt.tight_layout()

## 3 · The numbers

Per kind of switch: how many clean mice, and the mean of their `convergence_100_300` with its 95 % t interval. The
unit is the mouse; a mouse's switches of one kind are combined first. Below, every mouse.

In [ ]:
rows = []
for (design, kind), g in clean(animals).groupby(['design', 'switch_kind']):
    m, lo, hi, n = t_interval(g['convergence_100_300'])
    rows.append({'design': design, 'switch_kind': kind, 'mice': n, 'mean': m, 'low': lo, 'high': hi})
display(pd.DataFrame(rows).round(2))
clean(animals).pivot_table(index=['design', 'animal'], columns='switch_kind', values='convergence_100_300').round(2)

## 4 · Where the blocks end

`convergence_final` is measured over the last 250 trials of the block, days after the switch, so the slow drift of
the criterion is inside it: it helps the Hard-B → Hard-A switch and works against Hard-A → Hard-B. Per switch, the
early number beside the end of the block.

In [ ]:
blk = clean(wide[wide['design'] == 'blocked'])
blk = blk.assign(switch=blk['from_distribution'] + ' → ' + blk['to_distribution'])
blk[['animal', 'switch_idx', 'switch', 'convergence_100_300', 'convergence_final']].round(2)

## 5 · Overnight

`overnight.csv` (blocks): the criterion over the last 100 trials of a session against the first 100 of the next.
`toward_pre` signs the change by the direction of the switch: > 0 = back towards the previous distribution
overnight. Signed this way, noise in either window cannot set the sign, which it did when the end-of-session value
itself chose it.

In [ ]:
ov = clean(overnight[overnight['design'] == 'blocked']) if len(overnight) else overnight
if len(ov) and ov['toward_pre'].notna().any():
    per = ov.groupby('animal')['toward_pre'].median()
    m, lo, hi, n = t_interval(per)
    print(f'overnight change back towards the previous distribution, median per mouse: mean {m:+.3f} '
          f'(95 % t interval {lo:+.3f} to {hi:+.3f}, {n} mice)')
    display(ov.groupby('animal').agg(pairs=('toward_pre', 'count'), median_toward_pre=('toward_pre', 'median'),
                                     median_change=('delta', 'median')).round(3))
else:
    print('no overnight pairs in these runs')